# 03 DPC training and the pre-specified selection

The policy is a small MLP that maps the current state (SoC, previous voltages at the battery buses, total load, PV at the
four PV buses, latest ramp rate, previous action, time of day; **no forecast**) to P and Q of both batteries. It is
trained by backpropagation through a rollout of the differentiable grid model, with the safety layer in the loop.

Candidates differ in the terminal SoC term (weight 10, 0, 0.1, or a 30-70% dead zone). They were judged by a rule written
before the check: a candidate must (a) use active power, mean |P| ≥ 0.025 MW for each battery in each regime, and (b) keep
every end-of-day SoC inside 20-80%; among those, the lowest validation objective wins. The objective is the mean over
valid steps of the proposal's eq. (16) terms (losses, voltage deviation, effort, soft SoC and voltage penalties), averaged
over both regimes. Evaluation is a continuous closed loop over all validation steps with 15 solver sweeps.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
def candidate_rows(record):
    rows = []
    for name, c in record["candidates"].items():
        ch = record["checks"][name]
        min_p = min(min(c["regimes"][r]["mean_abs_p_mw"]) for r in REGIMES)
        eod = [c["regimes"][r][k] for r in REGIMES for k in ("end_of_day_soc_min", "end_of_day_soc_max")]
        rows.append({"candidate": name, "objective": fmt(c["objective"]),
                     "objective ×0.7": fmt(c["regimes"]["base_x0.7"]["objective"]),
                     "objective ×1.0": fmt(c["regimes"]["stress_x1.0"]["objective"]),
                     "min mean |P| (MW)": fmt(min_p), "(a) P used": fmt(ch["uses_active_power"]),
                     "end-of-day SoC range": f"{fmt(min(eod), 2)} - {fmt(max(eod), 2)}",
                     "(b) no drift": fmt(ch["soc_stays_in_range"]), "passes": fmt(ch["passes"])})
    return table(rows, index="candidate")

def reference_rows(record):
    names = {"volt_var_droop": "volt_var_droop (original, untuned)"}
    return table([{"reference": names.get(n, n), "objective": fmt(r["objective"]),
                   "objective ×0.7": fmt(r["regimes"]["base_x0.7"]["objective"]),
                   "objective ×1.0": fmt(r["regimes"]["stress_x1.0"]["objective"])} for n, r in record["references"].items()],
                 index="reference")

## Experiment 1: horizon 24 steps (2 h)

In [ ]:
e1 = load("m3_soc_variants_1000.json")
display(Markdown(provenance(e1, "m3_soc_variants_1000.json")))
n_pass = sum(ch["passes"] for ch in e1["checks"].values())
if n_pass == 0:
    display(Markdown(f"No candidate passed the pre-specified rule (0 of {len(e1['checks'])}), which is why Experiment 2 was added."))
display(candidate_rows(e1))
display(reference_rows(e1))
display(Markdown(f"Selected by the rule: **{e1['selected_by_rule']}**."))

## Experiment 2: horizons 72 and 144 steps (6 h and 12 h)

Experiment 2 was added **after** seeing Experiment 1, in which the policy used almost no active power. Its protocol,
including the same rule and thresholds, was written before any of its runs. Experiment 1 stays reported above.

In [ ]:
e2 = load("m3_exp2_soc_variants.json")
display(Markdown(provenance(e2, "m3_exp2_soc_variants.json")))
display(candidate_rows(e2))
display(Markdown(f"Selected by the rule: **{e2['selected_by_rule']}**."))

In [ ]:
sel = e2["selected_by_rule"]
r = e2["candidates"][sel]["regimes"]
min_p = min(min(r[reg]["mean_abs_p_mw"]) for reg in REGIMES)
display(Markdown(f"The selected candidate passes condition (a) narrowly: its minimum mean |P| over both batteries and both "
                 f"regimes is {min_p:.5f} MW against the 0.025 MW threshold. The threshold was not changed."))

### Training stability

Pre-specified rule: a run is unstable if a loss or gradient is not finite, or if the gradient norm before clipping is
above 10 on more than 5% of the steps after step 100.

In [ ]:
st = load("m3_exp2_stability.json")
rows = [{"candidate": n, "unstable": fmt(r[-1]["unstable"]), "max grad norm": fmt(r[-1]["max_grad_norm"], 2),
         "share after step 100 above 10": pct(r[-1]["share_of_steps_after_100_with_grad_norm_above_10"]),
         "truncated rerun": "no" if len(r) == 1 else "yes", "minutes": fmt(r[-1]["seconds"] / 60, 1)} for n, r in st.items()]
table(rows, index="candidate")

### Training curve of the selected candidate (seed 0)

In [ ]:
run = e2["candidates"][e2["selected_by_rule"]]["run_id"]
curves = pd.read_csv(RESULTS / "runs" / run / "curves.csv")
val = curves.dropna(subset=["val_objective"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(curves["step"], curves["train_objective"], color=SERIES[0], linewidth=1, label="training batch")
axes[0].plot(val["step"], val["val_objective"], color=SERIES[1], marker="o", markersize=5, label="256 validation windows")
axes[0].set_yscale("log"); axes[0].set_xlabel("training step"); axes[0].set_ylabel("training objective (own terms)")
axes[0].legend(frameon=False); axes[0].set_title("Objective", loc="left")
axes[1].plot(curves["step"], curves["grad_norm"], color=SERIES[0], linewidth=1)
axes[1].axhline(10, color=INK_2, linestyle="--", linewidth=1); axes[1].set_yscale("log")
axes[1].set_xlabel("training step"); axes[1].set_ylabel("gradient norm before clipping"); axes[1].set_title("Gradient norm", loc="left")
fig.tight_layout(); plt.show()
display(Markdown(f"Run `{run}`. The training objective here includes the terminal SoC term, so it is not the same number "
                 "as the selection objective above."))

## How the selected policy uses the battery (descriptive only)

These diagnostics were made after the selection and cannot change it. Validation part, seed 0.

In [ ]:
dg = load("m3_exp2_diagnostics_rerun.json")
display(Markdown(provenance(dg, "m3_exp2_diagnostics_rerun.json")))
d = dg["candidates"]["H144_dead_zone_30_70"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for i, (reg, label) in enumerate(REGIMES.items()):
    hp = np.array(d[reg]["hourly_mean_p_mw"])
    for b in range(hp.shape[1]):
        axes[0].plot(range(24), hp[:, b], color=SERIES[2 * i + b], label=f"battery {b + 1}, {label}")
    soc = np.array(d[reg]["soc_curve_mean"]).reshape(len(d[reg]["soc_curve_mean"]), -1)
    for b in range(soc.shape[1]):
        axes[1].plot(np.arange(len(soc)) * 24 / len(soc), soc[:, b], color=SERIES[2 * i + b], label=f"battery {b + 1}, {label}")
axes[0].axhline(0, color=INK_2, linewidth=1); axes[0].set_xlabel("hour of day"); axes[0].set_ylabel("mean P (MW), > 0 = discharge")
axes[0].set_title("Mean active power by hour", loc="left"); axes[0].legend(frameon=False, fontsize=8)
axes[1].set_xlabel("hour of day"); axes[1].set_ylabel("mean SoC"); axes[1].set_title("Mean SoC over the day", loc="left")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()
rows = [{"regime": label, "full days": d[reg]["full_days"],
         "throughput (MWh/day per battery)": fmt(float(np.mean(d[reg]["throughput_mwh_per_day_mean"])), 3),
         "equivalent full cycles per day": fmt(float(np.mean(d[reg]["equivalent_full_cycles_per_day_mean"])), 3),
         "mean daily SoC swing": fmt(float(np.mean(d[reg]["daily_soc_swing_mean"])), 3)} for reg, label in REGIMES.items()]
table(rows, index="regime")